# PS S6E4 - exp_20260409_028_stack_core4_screening

目的:
 - 018 / 024 / 025 / 026 の主力候補をスタック素材として整理する
 - 024 と 025 は競合枠、026 は別枠候補として扱う
 - simple average と ridge blend だけで first screening を行う

今回の候補セット:
 - set1 = [018, 024]
 - set2 = [018, 026]
 - set3 = [018, 024, 026]
 - set4 = [018, 025]
 - set5 = [018, 025, 026]

## Imports

In [1]:
import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260409_028_stack_core4_screening"
    EXP_NAME = "stack_core4_screening"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 42
    N_CLASSES = 3

    RIDGE_ALPHA = 1.0

    BIAS_GRID = [
        [1.0, 1.0, 1.0],
        [1.2, 1.2, 1.2],
        [1.5, 1.5, 1.5],
        [1.5, 1.3, 1.8],
        [1.7, 1.5, 2.3],
        [1.8, 1.5, 2.4],
    ]

## utility

In [3]:
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

def balanced_acc_score_mc(y_true, proba_or_pred):
    arr = np.asarray(proba_or_pred)
    if arr.ndim == 2:
        pred = np.argmax(arr, axis=1)
    else:
        pred = arr
    return float(balanced_accuracy_score(y_true, pred))

def one_hot(y, n_classes):
    y = np.asarray(y).astype(int)
    out = np.zeros((len(y), n_classes), dtype=float)
    out[np.arange(len(y)), y] = 1.0
    return out

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-15, None)
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum = np.where(row_sum == 0, 1.0, row_sum)
    return p / row_sum

def apply_class_weights_to_proba(proba, class_weights):
    class_weights = np.asarray(class_weights, dtype=float)
    adj = proba * class_weights[None, :]
    return normalize_proba(adj)

seed_everything(CFG.SEED)

## load target / submission template

In [4]:
train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)
sample_sub = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv")

target2idx = {v: i for i, v in enumerate(train_df[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}

y = train_df[CFG.TARGET_COL].map(target2idx).values.astype(int)

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
print("target mapping:", target2idx)

train shape: (630000, 21)
test shape : (270000, 20)
target mapping: {'Low': 0, 'Medium': 1, 'High': 2}


## model_dict load

In [5]:
model_dict = {
    "018": (
        np.load(CFG.NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
    ),
    "024": (
        np.load(CFG.NPY_PATH + "oof_xgb_digit_orderedte_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_xgb_digit_orderedte_min_proba_biased.npy"),
    ),
    "025": (
        np.load(CFG.NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"),
    ),
    "026": (
        np.load(CFG.NPY_PATH + "oof_realmlp_pair_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_realmlp_pair_te_min_proba_biased.npy"),
    ),
}

## class order fix

In [6]:
perm = [2, 0, 1]

for name in ["018", "024", "025", "026"]:
    oof_t, pred_t = model_dict[name]
    model_dict[name] = (oof_t[:, perm], pred_t[:, perm])

for name, (oof_arr, pred_arr) in model_dict.items():
    print(name)
    print("  oof shape:", oof_arr.shape, " row sum head:", np.round(oof_arr[:5].sum(axis=1), 6))
    print("  pred shape:", pred_arr.shape, " row sum head:", np.round(pred_arr[:5].sum(axis=1), 6))

018
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]
024
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]
025
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]
026
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]


## candidate sets

In [7]:
candidate_sets = {
    "set1_018_024": ["018", "024"],
    "set2_018_026": ["018", "026"],
    "set3_018_024_026": ["018", "024", "026"],
    "set4_018_025": ["018", "025"],
    "set5_018_025_026": ["018", "025", "026"],
}

candidate_sets

{'set1_018_024': ['018', '024'],
 'set2_018_026': ['018', '026'],
 'set3_018_024_026': ['018', '024', '026'],
 'set4_018_025': ['018', '025'],
 'set5_018_025_026': ['018', '025', '026']}

## simple average blend

In [8]:
def simple_average_blend(model_names, model_dict):
    oof_stack = np.mean([model_dict[m][0] for m in model_names], axis=0)
    pred_stack = np.mean([model_dict[m][1] for m in model_names], axis=0)
    oof_stack = normalize_proba(oof_stack)
    pred_stack = normalize_proba(pred_stack)
    return oof_stack, pred_stack

## ridge blend

In [9]:
def build_meta_features(model_names, model_dict, use_logit=False):
    oof_feats = []
    pred_feats = []

    for m in model_names:
        oof_p = model_dict[m][0].copy()
        pred_p = model_dict[m][1].copy()

        if use_logit:
            eps = 1e-15
            oof_p = np.log(np.clip(oof_p, eps, 1 - eps) / np.clip(1 - oof_p, eps, 1 - eps))
            pred_p = np.log(np.clip(pred_p, eps, 1 - eps) / np.clip(1 - pred_p, eps, 1 - eps))

        oof_feats.append(oof_p)
        pred_feats.append(pred_p)

    X_oof = np.concatenate(oof_feats, axis=1)
    X_pred = np.concatenate(pred_feats, axis=1)
    return X_oof, X_pred


def ridge_multiclass_blend(model_names, model_dict, y_true, alpha=1.0, use_logit=False):
    X_oof, X_pred = build_meta_features(model_names, model_dict, use_logit=use_logit)
    y_onehot = one_hot(y_true, CFG.N_CLASSES)

    pred_oof = np.zeros((len(y_true), CFG.N_CLASSES), dtype=float)
    pred_test = np.zeros((X_pred.shape[0], CFG.N_CLASSES), dtype=float)

    for c in range(CFG.N_CLASSES):
        reg = Ridge(alpha=alpha, random_state=CFG.SEED)
        reg.fit(X_oof, y_onehot[:, c])
        pred_oof[:, c] = reg.predict(X_oof)
        pred_test[:, c] = reg.predict(X_pred)

    pred_oof = normalize_proba(pred_oof)
    pred_test = normalize_proba(pred_test)
    return pred_oof, pred_test

## bias search helper

In [10]:
def search_best_bias_grid(oof_proba, y_true, bias_grid):
    best_score = balanced_acc_score_mc(y_true, oof_proba)
    best_weights = np.ones(CFG.N_CLASSES, dtype=float)

    for cw in bias_grid:
        cw = np.asarray(cw, dtype=float)
        adjusted = apply_class_weights_to_proba(oof_proba, cw)
        score = balanced_acc_score_mc(y_true, adjusted)
        if score > best_score:
            best_score = score
            best_weights = cw.copy()

    return best_weights, best_score

## run screening

In [11]:
results = []
blend_outputs = {}

for set_name, model_names in candidate_sets.items():
    print("=" * 100)
    print(set_name, model_names)
    print("=" * 100)

    # simple average
    avg_oof, avg_pred = simple_average_blend(model_names, model_dict)
    avg_raw_cv = balanced_acc_score_mc(y, avg_oof)
    avg_best_w, avg_biased_cv = search_best_bias_grid(avg_oof, y, CFG.BIAS_GRID)
    avg_oof_biased = apply_class_weights_to_proba(avg_oof, avg_best_w)
    avg_pred_biased = apply_class_weights_to_proba(avg_pred, avg_best_w)

    results.append({
        "set_name": set_name,
        "models": "+".join(model_names),
        "blend_type": "simple_average",
        "raw_cv_ba": avg_raw_cv,
        "biased_cv_ba": avg_biased_cv,
        "best_class_weights": avg_best_w.tolist(),
    })

    blend_outputs[(set_name, "simple_average")] = {
        "oof_raw": avg_oof,
        "pred_raw": avg_pred,
        "oof_biased": avg_oof_biased,
        "pred_biased": avg_pred_biased,
    }

    print(f"[simple_average] raw_cv={avg_raw_cv:.6f} | biased_cv={avg_biased_cv:.6f}")

    # ridge blend
    ridge_oof, ridge_pred = ridge_multiclass_blend(
        model_names=model_names,
        model_dict=model_dict,
        y_true=y,
        alpha=CFG.RIDGE_ALPHA,
        use_logit=False,
    )
    ridge_raw_cv = balanced_acc_score_mc(y, ridge_oof)
    ridge_best_w, ridge_biased_cv = search_best_bias_grid(ridge_oof, y, CFG.BIAS_GRID)
    ridge_oof_biased = apply_class_weights_to_proba(ridge_oof, ridge_best_w)
    ridge_pred_biased = apply_class_weights_to_proba(ridge_pred, ridge_best_w)

    results.append({
        "set_name": set_name,
        "models": "+".join(model_names),
        "blend_type": "ridge",
        "raw_cv_ba": ridge_raw_cv,
        "biased_cv_ba": ridge_biased_cv,
        "best_class_weights": ridge_best_w.tolist(),
    })

    blend_outputs[(set_name, "ridge")] = {
        "oof_raw": ridge_oof,
        "pred_raw": ridge_pred,
        "oof_biased": ridge_oof_biased,
        "pred_biased": ridge_pred_biased,
    }

    print(f"[ridge]          raw_cv={ridge_raw_cv:.6f} | biased_cv={ridge_biased_cv:.6f}")

set1_018_024 ['018', '024']
[simple_average] raw_cv=0.014670 | biased_cv=0.015444
[ridge]          raw_cv=0.979550 | biased_cv=0.979983
set2_018_026 ['018', '026']
[simple_average] raw_cv=0.014468 | biased_cv=0.015004
[ridge]          raw_cv=0.979102 | biased_cv=0.979267
set3_018_024_026 ['018', '024', '026']
[simple_average] raw_cv=0.014699 | biased_cv=0.015417
[ridge]          raw_cv=0.979484 | biased_cv=0.979974
set4_018_025 ['018', '025']
[simple_average] raw_cv=0.014345 | biased_cv=0.015015
[ridge]          raw_cv=0.979681 | biased_cv=0.979951
set5_018_025_026 ['018', '025', '026']
[simple_average] raw_cv=0.014375 | biased_cv=0.015124
[ridge]          raw_cv=0.979638 | biased_cv=0.980126


## results table

In [12]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["biased_cv_ba", "raw_cv_ba"], ascending=False).reset_index(drop=True)
display(results_df)

,set_name,models,blend_type,raw_cv_ba,biased_cv_ba,best_class_weights
0,set5_018_025_026,018+025+026,ridge,0.979638,0.980126,"[1.5, 1.3, 1.8]"
1,set1_018_024,018+024,ridge,0.979550,0.979983,"[1.8, 1.5, 2.4]"
2,set3_018_024_026,018+024+026,ridge,0.979484,0.979974,"[1.8, 1.5, 2.4]"
3,set4_018_025,018+025,ridge,0.979681,0.979951,"[1.7, 1.5, 2.3]"
4,set2_018_026,018+026,ridge,0.979102,0.979267,"[1.7, 1.5, 2.3]"
5,set1_018_024,018+024,simple_average,0.014670,0.015444,"[1.7, 1.5, 2.3]"
6,set3_018_024_026,018+024+026,simple_average,0.014699,0.015417,"[1.7, 1.5, 2.3]"
7,set5_018_025_026,018+025+026,simple_average,0.014375,0.015124,"[1.7, 1.5, 2.3]"
8,set4_018_025,018+025,simple_average,0.014345,0.015015,"[1.7, 1.5, 2.3]"
9,set2_018_026,018+026,simple_average,0.014468,0.015004,"[1.7, 1.5, 2.3]"


## save results summary

In [13]:
results_df.to_csv(CFG.SAVE_DIR / "screening_results.csv", index=False)

with open(CFG.SAVE_DIR / "screening_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("saved summary to:", CFG.SAVE_DIR)

saved summary to: /kaggle/working/exp_20260409_028_stack_core4_screening


## save top blend artifacts

In [14]:
best_row = results_df.iloc[0]
best_set = best_row["set_name"]
best_blend_type = best_row["blend_type"]

print("best_set =", best_set)
print("best_blend_type =", best_blend_type)

best_obj = blend_outputs[(best_set, best_blend_type)]

np.save(CFG.SAVE_DIR / f"oof_{best_set}_{best_blend_type}_proba.npy", best_obj["oof_raw"])
np.save(CFG.SAVE_DIR / f"pred_{best_set}_{best_blend_type}_proba.npy", best_obj["pred_raw"])
np.save(CFG.SAVE_DIR / f"oof_{best_set}_{best_blend_type}_proba_biased.npy", best_obj["oof_biased"])
np.save(CFG.SAVE_DIR / f"pred_{best_set}_{best_blend_type}_proba_biased.npy", best_obj["pred_biased"])

best_pred_idx = np.argmax(best_obj["pred_biased"], axis=1)
submission = sample_sub.copy()
submission[CFG.TARGET_COL] = pd.Series(best_pred_idx).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{best_set}_{best_blend_type}.csv", index=False)

display(submission.head())

best_set = set5_018_025_026
best_blend_type = ridge


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


## quick interpretation helper

In [15]:
simple_df = results_df[results_df["blend_type"] == "simple_average"].copy()
ridge_df = results_df[results_df["blend_type"] == "ridge"].copy()

print("=== simple average ranking ===")
display(simple_df)

print("=== ridge ranking ===")
display(ridge_df)

=== simple average ranking ===


,set_name,models,blend_type,raw_cv_ba,biased_cv_ba,best_class_weights
5,set1_018_024,018+024,simple_average,0.014670,0.015444,"[1.7, 1.5, 2.3]"
6,set3_018_024_026,018+024+026,simple_average,0.014699,0.015417,"[1.7, 1.5, 2.3]"
7,set5_018_025_026,018+025+026,simple_average,0.014375,0.015124,"[1.7, 1.5, 2.3]"
8,set4_018_025,018+025,simple_average,0.014345,0.015015,"[1.7, 1.5, 2.3]"
9,set2_018_026,018+026,simple_average,0.014468,0.015004,"[1.7, 1.5, 2.3]"


=== ridge ranking ===


,set_name,models,blend_type,raw_cv_ba,biased_cv_ba,best_class_weights
0,set5_018_025_026,018+025+026,ridge,0.979638,0.980126,"[1.5, 1.3, 1.8]"
1,set1_018_024,018+024,ridge,0.979550,0.979983,"[1.8, 1.5, 2.4]"
2,set3_018_024_026,018+024+026,ridge,0.979484,0.979974,"[1.8, 1.5, 2.4]"
3,set4_018_025,018+025,ridge,0.979681,0.979951,"[1.7, 1.5, 2.3]"
4,set2_018_026,018+026,ridge,0.979102,0.979267,"[1.7, 1.5, 2.3]"


## save meta note

In [16]:
meta_note = {
    "candidate_sets": candidate_sets,
    "ridge_alpha": CFG.RIDGE_ALPHA,
    "bias_grid": CFG.BIAS_GRID,
    "best_set": best_set,
    "best_blend_type": best_blend_type,
}

with open(CFG.SAVE_DIR / "meta_note.json", "w", encoding="utf-8") as f:
    json.dump(meta_note, f, ensure_ascii=False, indent=2)

print("done")

done
